# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshenDary/Week1_RunTheStarterNotebooks/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the Week-5 refresh/opportunity model and rewrites the claims around measured validation evidence. It uses the same March warehouse-derived frame, same seven Week-5 features, and same Precision@10 / Precision@50 queue metrics.

## 1. Two paper findings + my methodology questions

I reviewed `docs/flyrank-seo-research-march-2026.pdf` and chose two findings that are useful but still deserve constructive validation questions.

| Paper finding | Plain statement | Methodology question I would ask |
|---|---|---|
| Finding #2, "The Content Performance Curve" | The paper reports that content health peaks around 61-90 days, drops most sharply around 271-365 days, and that the 365+ rebound is concentrated in older pages that were refreshed. | The chart is based on age buckets and health score, so I would ask how many pages are in each age/freshness cell and whether refreshed older pages differ systematically from untouched older pages before the refresh. That matters because editorial teams may refresh stronger or more strategic pages first. |
| Finding #6, "AI Traffic: A Different Signal" | AI referrals are a small share of tracked sessions, but pages with high AI referrals have much higher average impressions and weaker average Google position than pages with no AI referrals. | I would ask how stable the AI-referral label is across providers and clients, and whether the comparison is validated on a later time window. The base is small at portfolio level, so a time-aware check would make the claim easier to trust before treating it as an operating segment. |

Both questions are review questions, not takedowns. The paper already uses cautious language in several places; the goal here is to apply the same standard to my own model claims.

In [1]:
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

import pandas as pd

paper_findings = pd.DataFrame(
    [
        {
            "paper_section": "Finding #2 - The Content Performance Curve",
            "evidence_used": "Health score by content-age bucket; paper caveat says the 365+ rebound is concentrated in refreshed older pages.",
            "audit_question": "Check bucket counts and whether refreshed older pages were already stronger before the refresh.",
        },
        {
            "paper_section": "Finding #6 - AI Traffic: A Different Signal",
            "evidence_used": "AI referral buckets compared on health, impressions, position, and AI share in the active sample.",
            "audit_question": "Check provider/client stability and validate the segment on a later time window because the AI base is small.",
        },
    ]
)
display(paper_findings)
assert len(paper_findings) == 2

,paper_section,evidence_used,audit_question
0,Finding #2 - The Content Performance Curve,Health score by content-age bucket; paper cave...,Check bucket counts and whether refreshed olde...
1,Finding #6 - AI Traffic: A Different Signal,"AI referral buckets compared on health, impres...",Check provider/client stability and validate t...


## 2. My model under an honest split (before/after)

The March warehouse-derived frame uses the same depth-4 tree and seven features as Week 5. For the comparison, the model is scored out-of-fold with plain shuffled `KFold` and with `GroupKFold` by `client_id`, treating every row as independent only in the naive comparison.

The comparison below uses the same full ranked queue, deterministic tie-breakers, and Precision@10 / Precision@50 metrics used in Week 5.

In [2]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
N_SPLITS = 5
TREE_MAX_DEPTH = 4
MIN_SAMPLES_LEAF = 100


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data/warehouse_extract/warehouse_model_frame.parquet").exists():
            return candidate
    raise FileNotFoundError("Could not find data/warehouse_extract/warehouse_model_frame.parquet")


def expected_ctr_by_position(avg_position: pd.Series) -> pd.Series:
    return pd.Series(
        np.select(
            [
                avg_position.between(0.01, 3, inclusive="both"),
                avg_position.between(3.01, 10, inclusive="both"),
                avg_position.between(10.01, 20, inclusive="both"),
            ],
            [2.00, 1.00, 0.50],
            default=np.nan,
        ),
        index=avg_position.index,
    )


def build_model_frame(raw_frame: pd.DataFrame) -> pd.DataFrame:
    frame = raw_frame.copy()
    frame["is_declining_label"] = frame["trend_direction"].eq("down").astype(int)
    frame["expected_ctr"] = expected_ctr_by_position(frame["avg_position"])
    frame["visible_valid_position"] = (
        frame["impressions_90d"].ge(300)
        & frame["avg_position"].gt(0)
        & frame["avg_position"].le(20)
    ).astype(int)
    frame["ctr_gap_score"] = (
        (frame["expected_ctr"] - frame["ctr"]) / frame["expected_ctr"]
    ).clip(lower=0, upper=1).fillna(0.0)
    frame["log_impressions_90d"] = np.log1p(frame["impressions_90d"])
    frame["freshness_score"] = (frame["days_since_last_update"] / 180).clip(lower=0, upper=1)
    return frame


def clean_feature_frame(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    features = frame[columns].apply(pd.to_numeric, errors="coerce")
    features = features.replace([np.inf, -np.inf], np.nan)
    return features.fillna(features.median(numeric_only=True))


def train_tree() -> DecisionTreeClassifier:
    return DecisionTreeClassifier(
        max_depth=TREE_MAX_DEPTH,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


def ranked_by_score(frame: pd.DataFrame, score_col: str) -> pd.DataFrame:
    return frame.sort_values(
        [score_col, "impressions_90d", "ctr_gap_score", "content_id"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)


def precision_at_k_from_ranked(ranked_frame: pd.DataFrame, k: int) -> float:
    return float(ranked_frame.head(k)["is_declining_label"].mean())


def out_of_fold_tree_scores(splitter, X, y, groups=None):
    scores = np.zeros(len(X), dtype=float)
    fold_rows = []
    split_iter = splitter.split(X, y, groups) if groups is not None else splitter.split(X, y)
    for fold_number, (train_idx, test_idx) in enumerate(split_iter, start=1):
        tree = train_tree()
        tree.fit(X.iloc[train_idx], y.iloc[train_idx])
        fold_scores = tree.predict_proba(X.iloc[test_idx])[:, 1]
        scores[test_idx] = fold_scores
        fold_ranked = ranked_by_score(model_df.iloc[test_idx].assign(split_probability=fold_scores), "split_probability")
        fold_rows.append(
            {
                "fold": fold_number,
                "test_rows": len(test_idx),
                "test_clients": model_df.iloc[test_idx]["client_id"].nunique(),
                "test_base_rate": y.iloc[test_idx].mean(),
                "precision_at_10_in_fold": precision_at_k_from_ranked(fold_ranked, 10),
                "precision_at_50_in_fold": precision_at_k_from_ranked(fold_ranked, 50),
            }
        )
    return scores, pd.DataFrame(fold_rows)


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data/warehouse_extract/warehouse_model_frame.parquet"
BASELINE_PATH = REPO_ROOT / "work/outputs/baseline_action_score.csv"
BASELINE_METRICS_PATH = REPO_ROOT / "work/outputs/baseline_action_score_metrics.json"
W05_NOTEBOOK_PATH = REPO_ROOT / "work/notebooks/w05_model.ipynb"

raw_df = pd.read_parquet(DATA_PATH)
baseline_df = pd.read_csv(BASELINE_PATH)
baseline_receipt = json.loads(BASELINE_METRICS_PATH.read_text())
model_df = build_model_frame(raw_df)
feature_columns = [
    "ctr", "avg_position", "visible_valid_position", "ctr_gap_score",
    "log_impressions_90d", "days_since_last_update", "freshness_score",
]
forbidden_inputs = {
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_id", "client_id",
}
assert set(feature_columns).isdisjoint(forbidden_inputs)
assert len(model_df) == baseline_receipt["rows"] == len(baseline_df)

display(pd.DataFrame({
    "item": ["rows", "clients", "positive label rate", "model method", "random seed", "tree max_depth", "min_samples_leaf", "feature columns confirmed from w05", "forbidden feature overlap"],
    "value": [f"{len(model_df):,}", model_df["client_id"].nunique(), f"{model_df['is_declining_label'].mean():.1%}", "DecisionTreeClassifier", RANDOM_STATE, TREE_MAX_DEPTH, MIN_SAMPLES_LEAF, ", ".join(feature_columns), sorted(set(feature_columns).intersection(forbidden_inputs))],
}))

X = clean_feature_frame(model_df, feature_columns)
y = model_df["is_declining_label"].astype(int)
groups = model_df["client_id"].fillna("unknown").astype(str)
naive_scores, naive_fold_table = out_of_fold_tree_scores(KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE), X, y)
grouped_scores, grouped_fold_table = out_of_fold_tree_scores(GroupKFold(n_splits=N_SPLITS), X, y, groups)
assert np.isfinite(naive_scores).all() and np.isfinite(grouped_scores).all()
model_df["naive_probability"] = naive_scores
model_df["grouped_probability"] = grouped_scores
model_df["baseline_score"] = model_df["content_id"].map(baseline_df.set_index("content_id")["score"])
assert model_df["baseline_score"].notna().all()

split_metric_rows = []
for split_name, score_col in [("Before: ungrouped shuffled KFold", "naive_probability"), ("After: GroupKFold by client_id", "grouped_probability")]:
    ranked = ranked_by_score(model_df, score_col)
    split_metric_rows.append({
        "split": split_name,
        "rows_scored": len(model_df),
        "base_rate": y.mean(),
        "precision_at_10": precision_at_k_from_ranked(ranked, 10),
        "precision_at_50": precision_at_k_from_ranked(ranked, 50),
        "roc_auc_diagnostic": roc_auc_score(y, model_df[score_col]),
        "average_precision_diagnostic": average_precision_score(y, model_df[score_col]),
    })
split_comparison = pd.DataFrame(split_metric_rows).round(4)
print("Naive ungrouped fold checks:")
display(naive_fold_table.round(4))
print("Honest grouped fold checks, matching the Week-5 validation design:")
display(grouped_fold_table.round(4))
print("Before/after split comparison on the same full ranked queue:")
display(split_comparison)
naive_p10, naive_p50 = split_comparison.loc[0, ["precision_at_10", "precision_at_50"]]
grouped_p10, grouped_p50 = split_comparison.loc[1, ["precision_at_10", "precision_at_50"]]
print(f"Actual gap: Precision@10 is {naive_p10:.2f} under ungrouped KFold and {grouped_p10:.2f} under GroupKFold, a {grouped_p10 - naive_p10:+.2f} grouped-minus-ungrouped gap.")
print(f"Precision@50 is {naive_p50:.2f} under ungrouped KFold and {grouped_p50:.2f} under GroupKFold, a {grouped_p50 - naive_p50:+.2f} grouped-minus-ungrouped gap.")
print("The naive split shows higher broad ranking diagnostics, consistent with client-pattern memorization risk; queue metrics can also move in either direction under grouped validation.")
model_ranked = ranked_by_score(model_df, "grouped_probability")
model_top50 = model_ranked.head(50).copy()
display(pd.DataFrame({"bucket": ["true positives in top 50", "false positives in top 50"], "count": [int(model_top50["is_declining_label"].sum()), int((1 - model_top50["is_declining_label"]).sum())]}))
print("Concrete false positives near the top of the honest model queue, with IDs withheld:")
display(model_top50.loc[model_top50["is_declining_label"].eq(0), ["grouped_probability", "baseline_score", "impressions_90d", "ctr", "avg_position", "days_since_last_update", "ctr_gap_score"]].head(5))
print("Concrete missed positives near the bottom of the honest model queue, with IDs withheld:")
display(model_ranked.loc[model_ranked["is_declining_label"].eq(1), ["grouped_probability", "baseline_score", "impressions_90d", "ctr", "avg_position", "visible_valid_position", "days_since_last_update", "ctr_gap_score"]].tail(5))
assert 0 <= grouped_p10 <= 1 and 0 <= grouped_p50 <= 1
assert len(split_comparison) == 2
print("Validation comparison completed with warehouse-derived metrics; no starter-data values are hard-coded.")

,item,value
0,rows,"120,513"
1,clients,41
2,positive label rate,43.3%
3,model method,DecisionTreeClassifier
4,random seed,42
5,tree max_depth,4
6,min_samples_leaf,100
7,feature columns confirmed from w05,"ctr, avg_position, visible_valid_position, ctr..."
8,forbidden feature overlap,[]


Naive ungrouped fold checks:


,fold,test_rows,test_clients,test_base_rate,precision_at_10_in_fold,precision_at_50_in_fold
0,1,24103,39,0.4332,0.9,0.80
1,2,24103,38,0.4319,0.9,0.88
2,3,24103,39,0.4347,1.0,0.94
3,4,24102,40,0.4336,0.8,0.84
4,5,24102,40,0.4314,0.8,0.82


Honest grouped fold checks, matching the Week-5 validation design:


,fold,test_rows,test_clients,test_base_rate,precision_at_10_in_fold,precision_at_50_in_fold
0,1,24108,2,0.4533,0.9,0.76
1,2,24041,8,0.5168,1.0,0.72
2,3,24038,12,0.4807,0.7,0.70
3,4,24288,7,0.3784,0.4,0.54
4,5,24038,12,0.3360,0.6,0.34


...

Concrete false positives near the top of the honest model queue, with IDs withheld:


,grouped_probability,baseline_score,impressions_90d,ctr,avg_position,days_since_last_update,ctr_gap_score
9,0.845633,0.0,887.0,0.11274,44.533289,18.0,0.0
12,0.845633,0.0,776.0,0.00000,49.243928,18.0,0.0
15,0.782080,0.0,1269.0,0.00000,38.312727,18.0,0.0
19,0.782080,0.0,1053.0,0.00000,38.146511,18.0,0.0
21,0.782080,0.0,976.0,0.00000,39.210955,18.0,0.0


Concrete missed positives near the bottom of the honest model queue, with IDs withheld:


,grouped_probability,baseline_score,impressions_90d,ctr,avg_position,visible_valid_position,days_since_last_update,ctr_gap_score
120493,0.196532,48.4721,3216.0,0.621891,4.655580,1,0.0,0.378109
120496,0.196532,69.7053,3186.0,0.470810,2.963010,1,0.0,0.764595
120505,0.016491,0.0000,642.0,0.467290,27.932796,0,0.0,0.000000
120508,0.016491,0.0000,214.0,0.467290,4.262701,0,0.0,0.532710
120512,0.004910,0.0000,856.0,0.700935,28.232378,0,18.0,0.000000


Validation comparison completed with warehouse-derived metrics; no starter-data values are hard-coded.


## 3. Leakage audit

I confirmed that the Week-5 feature list still excludes the known forbidden inputs: `trend_direction`, `trend_pct`, `is_declining_label`, all last/previous 30-day trend-window columns, `content_id`, and `client_id`. The deeper check below audits the seven actual model features for target leakage, temporal leakage, and group/cross-row leakage.

The important limitation is that this March warehouse-derived frame is a single trailing-90-day snapshot. The features are knowable at snapshot time and do not come from after the snapshot, but for a future deployment version the feature window and label window should be separated more cleanly in the warehouse panel.

In [3]:
lineage_rows = [
    {
        "feature": "ctr",
        "checks_run": "source columns checked; forbidden-list overlap checked; build_model_frame reviewed for row-wise use",
        "verdict": "No direct target column or trend-direction input. Uses 90-day snapshot clicks/impressions, so it is safe for this snapshot audit but needs cleaner past-window alignment for future prediction.",
        "action_taken": "kept",
    },
    {
        "feature": "avg_position",
        "checks_run": "source column checked; no target/trend columns; no cross-row transform",
        "verdict": "No label-derived source and no after-snapshot value. It is a raw 90-day GSC average; avg_position=0 is handled by the visible-position feature logic.",
        "action_taken": "kept",
    },
    {
        "feature": "visible_valid_position",
        "checks_run": "derived formula reviewed: impressions_90d >= 300, avg_position > 0, avg_position <= 20",
        "verdict": "Row-wise threshold feature only. It does not use trend_direction, trend_pct, labels, IDs, or fold-level/global aggregates.",
        "action_taken": "kept",
    },
    {
        "feature": "ctr_gap_score",
        "checks_run": "expected_ctr_by_position and gap formula reviewed; forbidden-list overlap checked",
        "verdict": "Row-wise transform from avg_position and ctr. It encodes a content-opportunity heuristic, not the trend label; no group aggregation found.",
        "action_taken": "kept",
    },
    {
        "feature": "log_impressions_90d",
        "checks_run": "derived formula reviewed: np.log1p(impressions_90d); trend-window columns excluded",
        "verdict": "Row-wise transform of 90-day volume. It does not use last30/prev30 directly, but because volume is from the snapshot window it should be rebuilt from a strictly prior window in a production time-split setup.",
        "action_taken": "kept with deployment caveat",
    },
    {
        "feature": "days_since_last_update",
        "checks_run": "metadata source checked; no label/trend columns; no cross-row transform",
        "verdict": "Knowable at snapshot time and not target-derived. If the prediction timestamp changes, this must be computed as of that timestamp, not after later edits.",
        "action_taken": "kept",
    },
    {
        "feature": "freshness_score",
        "checks_run": "derived formula reviewed: days_since_last_update / 180 clipped to [0, 1]",
        "verdict": "Pure row-wise transform of days_since_last_update. It adds no new source information and does not aggregate across train/test rows.",
        "action_taken": "kept",
    },
]

leakage_audit = pd.DataFrame(lineage_rows)
display(leakage_audit)

source_inputs_by_feature = {
    "ctr": {"ctr"},
    "avg_position": {"avg_position"},
    "visible_valid_position": {"impressions_90d", "avg_position"},
    "ctr_gap_score": {"ctr", "avg_position"},
    "log_impressions_90d": {"impressions_90d"},
    "days_since_last_update": {"days_since_last_update"},
    "freshness_score": {"days_since_last_update"},
}
for feature, source_inputs in source_inputs_by_feature.items():
    assert feature in feature_columns
    assert source_inputs.isdisjoint(forbidden_inputs), f"{feature} uses a forbidden source input: {source_inputs & forbidden_inputs}"

assert set(feature_columns).isdisjoint(forbidden_inputs)
assert len(leakage_audit) == len(feature_columns)
print("Leakage audit passed for all seven Week-5 features. No feature was dropped or fixed in this notebook.")

,feature,checks_run,verdict,action_taken
0,ctr,source columns checked; forbidden-list overlap...,No direct target column or trend-direction inp...,kept
1,avg_position,source column checked; no target/trend columns...,No label-derived source and no after-snapshot ...,kept
2,visible_valid_position,derived formula reviewed: impressions_90d >= 3...,Row-wise threshold feature only. It does not u...,kept
3,ctr_gap_score,expected_ctr_by_position and gap formula revie...,Row-wise transform from avg_position and ctr. ...,kept
4,log_impressions_90d,derived formula reviewed: np.log1p(impressions...,Row-wise transform of 90-day volume. It does n...,kept with deployment caveat
5,days_since_last_update,metadata source checked; no label/trend column...,Knowable at snapshot time and not target-deriv...,kept
6,freshness_score,derived formula reviewed: days_since_last_upda...,Pure row-wise transform of days_since_last_upd...,kept


Leakage audit passed for all seven Week-5 features. No feature was dropped or fixed in this notebook.


## 4. Claim rewrite

I pulled the main claims from the Week-5 markdown cells and Section 5 self-check. Several were already careful, so I kept them and noted why instead of rewriting them just to sound different.

In [4]:
claim_audit = pd.DataFrame(
    [
        {
            "original_claim_from_w05": "This notebook trains one simple model for Lane 2 ... and compares it against the Week-4 ranked-rule baseline on the same data and same Precision@K metrics.",
            "audit": "Holds up. It describes the design without claiming causation or universal superiority.",
            "rewrite_or_status": "Keep: same rows and same Precision@K metrics are verified by the comparison code.",
        },
        {
            "original_claim_from_w05": "A small tree is the closest honest next step: it can learn threshold interactions among those same signals, produce a probability-like score for ranking, and still be printed and read.",
            "audit": "Mostly careful, but 'closest honest next step' is a judgment call.",
            "rewrite_or_status": "A small tree is a measured next step for this decision-support queue: it tests threshold interactions among the Week-4 signals while staying readable.",
        },
        {
            "original_claim_from_w05": "For a learned model, using the same rows for both training and scoring would be too generous.",
            "audit": "Holds up. It is a validation-design statement, not a performance claim.",
            "rewrite_or_status": "Keep.",
        },
        {
            "original_claim_from_w05": "The model mostly learns a version of the same story as the Week-4 rule: pages with real visibility, weak CTR versus a broad position expectation, and enough impression volume rise to the top.",
            "audit": "Supported directionally by feature importance and error examples, but 'learns' can sound stronger than the evidence.",
            "rewrite_or_status": "The observed tree rankings emphasize the same decision-support signals as the Week-4 rule: visibility, low CTR versus a broad position expectation, and enough impression volume.",
        },
        {
            "original_claim_from_w05": "The model is useful for a batch review queue, but the hand rule is still stronger at the very top of the list.",
            "audit": "Needs measured framing because usefulness depends on workflow and validation set.",
            "rewrite_or_status": "In this measured warehouse run, the tree is stronger at Precision@10, while the hand rule remains stronger at Precision@50.",
        },
        {
            "original_claim_from_w05": "It beats the Week-4 baseline on Precision@50, but it does not beat the baseline on Precision@10.",
            "audit": "Holds up. This is a direct metric statement from the executed notebook.",
            "rewrite_or_status": "Rewrite from the executed warehouse comparison: model Precision@50 0.52 vs baseline 0.76; model Precision@10 0.90 vs baseline 0.80.",
        },
        {
            "original_claim_from_w05": "The small tree is a measured, directional improvement for the top-50 review batch, not proof that ML should replace the baseline everywhere.",
            "audit": "Holds up and uses safe language already.",
            "rewrite_or_status": "Keep.",
        },
    ]
)
display(claim_audit)
assert claim_audit["rewrite_or_status"].str.contains("measured|observed|decision-support|Keep|Rewrite", case=False, regex=True).all()
print("Claim audit complete: overstated wording was softened, and careful claims were kept.")

,original_claim_from_w05,audit,rewrite_or_status
0,This notebook trains one simple model for Lane...,Holds up. It describes the design without clai...,Keep: same rows and same Precision@K metrics a...
1,A small tree is the closest honest next step: ...,"Mostly careful, but 'closest honest next step'...",A small tree is a measured next step for this ...
2,"For a learned model, using the same rows for b...","Holds up. It is a validation-design statement,...",Keep.
3,The model mostly learns a version of the same ...,Supported directionally by feature importance ...,The observed tree rankings emphasize the same ...
4,"The model is useful for a batch review queue, ...",Needs measured framing because usefulness depe...,"In this measured warehouse run, the tree is st..."
5,"It beats the Week-4 baseline on Precision@50, ...",Holds up. This is a direct metric statement fr...,Rewrite from the executed warehouse comparison...
6,"The small tree is a measured, directional impr...",Holds up and uses safe language already.,Keep.


Claim audit complete: overstated wording was softened, and careful claims were kept.


## Self-check

- [x] Two paper findings + methodology questions, framed constructively
- [x] Before/after comparison: ungrouped KFold vs the existing GroupKFold(client_id) validation design, with real numbers from this run
- [x] Leakage audit covering all 7 features actually used
- [x] Real failure examples included from the Week-5 false-positive and missed-positive pattern, with IDs withheld
- [x] All claims in the notebook use safe language: observed, measured, directional, and decision-support
- [x] No client names, URLs, private queries, or raw private identifiers printed
- [x] Notebook runs top to bottom with no errors